# White balancing 
White balancing adjusts the colors of an image to make them appear more accurate or natural. It corrects the image so that objects which are physically white in color appear white in the picture. white balancing algorithm: 


## Gray World Assumption
One common approach is the gray world assumption, which posits that the average of all colors in a scene is gray. By adjusting the intensities of the R, G, and B channels so that their averages are equal, the algorithm can neutralize the color cast, aiming to achieve a color balance.
## White Patch

Let's denote the maximum values found in the image for the red, green, and blue channels as $ R_{\text{max}} $, $ G_{\text{max}} $, and $ B_{\text{max}} $ respectively. Also, let $ R_{\text{target}} $, $ G_{\text{target}} $, and $ B_{\text{target}} $ be the target values for white, which would typically be the maximum value that can be represented in the image (e.g., 255 in an 8-bit image).

1. **Determine the Adjustment Factors**: 

The adjustment factors for each channel are calculated as follows:

$
\text{Adjustment factor for R (}A_R\text{)} = \frac{R_{\text{target}}}{R_{\text{max}}}
$

$
\text{Adjustment factor for G (}A_G\text{)} = \frac{G_{\text{target}}}{G_{\text{max}}}
$

$
\text{Adjustment factor for B (}A_B\text{)} = \frac{B_{\text{target}}}{B_{\text{max}}}
$

2. **Apply the Correction**: 

For each pixel in the image with RGB values of $ (R, G, B) $, the adjusted values $ (R', G', B') $ are calculated as:

$
R' = R \times A_R
$

$
G' = G \times A_G
$

$
B' = B \times A_B
$

3. **Normalization (if necessary)**: 

After applying these adjustments, it's crucial to ensure that none of the new RGB values exceeds the maximum possible value (e.g., 255 in an 8-bit image). If any $ R' $, $ G' $, or $ B' $ is greater than the target maximum, you may need to scale the entire image down to fit within the valid range.

These equations succinctly describe the process of adjusting an image's white balance using the White Patch method. By applying these adjustments, the brightest points in the image are made to match the target white values, theoretically neutralizing any color cast and achieving a more natural color balance.


To apply the White Patch method for white balancing using OpenCV, you would typically follow these steps to adjust the image based on the maximum values found in each color channel. Here is a step-by-step breakdown and corresponding Python code using OpenCV:

1. **Load the Image**: First, you need to load the image for which you want to perform white balancing.

2. **Find the Maximum Values**: Identify the maximum values in the R, G, and B channels of the image.

3. **Calculate the Adjustment Factors**: Determine the scale factors for each channel based on the maximum value that should represent white (typically 255 for an 8-bit image).

4. **Apply the Correction**: Scale each pixel's color values accordingly.



In [1]:
import cv2
import numpy as np

# Load an image
image = cv2.imread('path/to/your/image.jpg')

# Ensure the image is in float type for processing
image = np.float32(image)

# Find the max values for each channel
max_vals = np.amax(image, axis=(0, 1))

# Calculate the scale factors
scale_factors = 255 / max_vals

# Apply the scale factors to each channel
corrected_image = image * scale_factors

# Clip values to the range [0, 255] and convert back to uint8
corrected_image = np.clip(corrected_image, 0, 255).astype(np.uint8)

# Save or display the corrected image
cv2.imwrite('path/to/save/corrected_image.jpg', corrected_image)
# cv2.imshow('Corrected Image', corrected_image)
# cv2.waitKey(0)
# cv2.destroyAllWindows()


[ WARN:0@0.199] global loadsave.cpp:248 findDecoder imread_('path/to/your/image.jpg'): can't open/read file: check file path/integrity


AxisError: axis 0 is out of bounds for array of dimension 0


In this script:

- The image is loaded and converted to a floating-point type to maintain precision during processing.
- `np.amax` is used to find the maximum values across the image for each channel.
- The scale factors for each channel are computed based on the assumption that the maximum value should map to 255.
- These scale factors are then applied to each pixel, and the image is scaled accordingly.
- Finally, the resulting image is clipped to ensure pixel values are within the acceptable range and converted back to an 8-bit format before being saved or displayed.


In [2]:
import cv2
import numpy as np

# Initialize a list to store the coordinates of the corners
points = []

def click_event(event, x, y, flags, params):
    # If the left mouse button was clicked, record the point
    if event == cv2.EVENT_LBUTTONDOWN:
        points.append((x, y))
        
        # Draw a circle at the clicked point for visual feedback
        cv2.circle(image, (x, y), 5, (0, 255, 0), -1)
        cv2.imshow('image', image)

# Load the image
image = cv2.imread('path/to/your/image.jpg')

# Display the image and set up the mouse callback function
cv2.imshow('image', image)
cv2.setMouseCallback('image', click_event)

# Wait until the user has clicked four points
while len(points) < 4:
    cv2.waitKey(1)

cv2.destroyAllWindows()

# After the four points are selected, define the ROI as the polygon bounded by these points
# Create a mask with the same size as the image, filled with zeros (black)
mask = np.zeros(image.shape[:2], dtype=np.uint8)

# Define the points of the polygon (the four clicked points)
roi_corners = np.array([points], dtype=np.int32)

# Fill the polygon white in the mask
cv2.fillPoly(mask, roi_corners, 255)

# Apply the mask to get the ROI
roi = cv2.bitwise_and(image, image, mask=mask)

# Now, calculate the average color in the ROI
# We need to exclude black color from averaging, since it's the masked out area
mask_indices = np.where(mask == 255)
avg_color = np.mean(image[mask_indices[0], mask_indices[1]], axis=0)

# Calculate the scaling factors
scaling_factors = 255 / avg_color

# Apply the white balancing
corrected_image = np.clip(image * scaling_factors, 0, 255).astype(np.uint8)

# Display the corrected image
cv2.imshow('White Balanced Image', corrected_image)
cv2.waitKey(0)
cv2.destroyAllWindows()


[ WARN:0@237.414] global loadsave.cpp:248 findDecoder imread_('path/to/your/image.jpg'): can't open/read file: check file path/integrity


error: OpenCV(4.9.0) /home/conda/feedstock_root/build_artifacts/libopencv_1708670056696/work/modules/highgui/src/window.cpp:971: error: (-215:Assertion failed) size.width>0 && size.height>0 in function 'imshow'
